# Feature engineering

Notebook dédié à la création et sélection des variables.

In [1]:
# Cellule 1 — chargement
import pandas as pd
import numpy as np

dataset_final = pd.read_csv("../data/processed/dataset_final.csv")
print(f"Shape initial : {dataset_final.shape}")

Shape initial : (220, 83)


In [2]:
# Cellule 2 — précipitations cumulées par saison
# Bénin Sud : deux saisons des pluies (mars-juillet) et (septembre-novembre)
# Bénin Nord : une saison des pluies (mai-octobre)

# Saison 1 : mars à juillet
cols_s1 = [f"precip_{m:02d}_mm" for m in [3, 4, 5, 6, 7]]
dataset_final["precip_saison1_mm"] = dataset_final[cols_s1].sum(axis=1)

# Saison 2 : septembre à novembre
cols_s2 = [f"precip_{m:02d}_mm" for m in [9, 10, 11]]
dataset_final["precip_saison2_mm"] = dataset_final[cols_s2].sum(axis=1)

# Saison de croissance principale : mai à octobre
cols_croissance = [f"precip_{m:02d}_mm" for m in [5, 6, 7, 8, 9, 10]]
dataset_final["precip_croissance_mm"] = dataset_final[cols_croissance].sum(axis=1)

print("Précipitations saisonnières créées :")
print(dataset_final[["departement", "annee", "precip_saison1_mm",
                       "precip_saison2_mm", "precip_croissance_mm"]].head(5))

Précipitations saisonnières créées :
  departement  annee  precip_saison1_mm  precip_saison2_mm  \
0     Atacora   2003             680.76             355.90   
1     Alibori   2003             425.58             175.03   
2  Atlantique   2003             733.53             450.54   
3      Borgou   2003             540.58             281.40   
4    Collines   2003             757.34             480.78   

   precip_croissance_mm  
0               1278.09  
1                851.19  
2                948.42  
3               1049.19  
4               1362.93  


Les valeurs sont cohérentes et réalistes. Atlantique et Collines ont plus de pluie que Alibori, ce qui confirme le gradient Nord-Sud. 

In [3]:
# Cellule 3 — amplitude thermique mensuelle (temp_max - temp_min)
for m in range(1, 13):
    dataset_final[f"amplitude_{m:02d}_c"] = (
        dataset_final[f"temp_max_{m:02d}_c"] - dataset_final[f"temp_min_{m:02d}_c"]
    )

# Amplitude thermique annuelle moyenne
cols_amplitude = [f"amplitude_{m:02d}_c" for m in range(1, 13)]
dataset_final["amplitude_annuelle_c"] = dataset_final[cols_amplitude].mean(axis=1)

print("Amplitudes thermiques créées :")
print(dataset_final[["departement", "annee"] + cols_amplitude[:4]].head(3))

Amplitudes thermiques créées :
  departement  annee  amplitude_01_c  amplitude_02_c  amplitude_03_c  \
0     Atacora   2003           23.91           22.40           21.67   
1     Alibori   2003           25.68           23.82           22.47   
2  Atlantique   2003           12.25            7.97            7.39   

   amplitude_04_c  
0           21.20  
1           24.27  
2            7.33  


L'Alibori et l'Atacora au Nord ont des amplitudes thermiques très élevées (23-25°C d'écart entre jour et nuit) alors qu'Atlantique au Sud n'a que 7-12°C d'écart, typique d'un climat côtier humide.

In [4]:
# Cellule 4 — stress thermique
# Seuil critique pour le maïs : température max > 35°C nuit la pollinisation
for m in range(1, 13):
    dataset_final[f"stress_thermique_{m:02d}"] = (
        dataset_final[f"temp_max_{m:02d}_c"] > 35
    ).astype(int)

# Nombre de mois avec stress thermique dans l'année
cols_stress = [f"stress_thermique_{m:02d}" for m in range(1, 13)]
dataset_final["nb_mois_stress_thermique"] = dataset_final[cols_stress].sum(axis=1)

print("Stress thermique créé :")
print(dataset_final.groupby("departement")["nb_mois_stress_thermique"].mean().round(1).sort_values(ascending=False))

Stress thermique créé :
departement
Alibori       9.1
Borgou        5.8
Donga         5.3
Atacora       4.8
Couffo        3.6
Collines      3.4
Zou           1.4
Plateau       0.7
Mono          0.6
Atlantique    0.0
Ouémé         0.0
Name: nb_mois_stress_thermique, dtype: float64


C:\Users\pc\AppData\Local\Temp\ipykernel_21956\2904831333.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset_final[f"stress_thermique_{m:02d}"] = (
C:\Users\pc\AppData\Local\Temp\ipykernel_21956\2904831333.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset_final[f"stress_thermique_{m:02d}"] = (
C:\Users\pc\AppData\Local\Temp\ipykernel_21956\2904831333.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider 

Alibori a 9.1 mois de stress thermique en moyenne, c'est le département le plus chaud du Nord. Atlantique et Ouémé ont 0, zones côtières fraîches. Ça confirme ce qu'on observait.

In [5]:
# Cellule 5 — variable de tendance temporelle
# Permet au modèle de capturer une éventuelle évolution des rendements dans le temps
dataset_final["annee_normalisee"] = (
    dataset_final["annee"] - dataset_final["annee"].min()
) / (dataset_final["annee"].max() - dataset_final["annee"].min())

print("Variable temporelle créée :")
print(dataset_final[["annee", "annee_normalisee"]].drop_duplicates().head(5))

Variable temporelle créée :
    annee  annee_normalisee
0    2003          0.000000
11   2004          0.052632
22   2005          0.105263
33   2006          0.157895
44   2007          0.210526


C:\Users\pc\AppData\Local\Temp\ipykernel_21956\3743317245.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset_final["annee_normalisee"] = (


In [6]:
# Cellule 6 — encodage du département
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
dataset_final["departement_encode"] = le.fit_transform(dataset_final["departement"])

# Sauvegarder le mapping pour l'interface web plus tard
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Encodage des départements :")
print(mapping)

Encodage des départements :
{'Alibori': np.int64(0), 'Atacora': np.int64(1), 'Atlantique': np.int64(2), 'Borgou': np.int64(3), 'Collines': np.int64(4), 'Couffo': np.int64(5), 'Donga': np.int64(6), 'Mono': np.int64(7), 'Ouémé': np.int64(8), 'Plateau': np.int64(9), 'Zou': np.int64(10)}


C:\Users\pc\AppData\Local\Temp\ipykernel_21956\2002790002.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset_final["departement_encode"] = le.fit_transform(dataset_final["departement"])


Alibori    → 0
Atacora    → 1
Atlantique → 2
Borgou     → 3
Collines   → 4
Couffo     → 5
Donga      → 6
Mono       → 7
Ouémé      → 8
Plateau    → 9
Zou        → 10

In [7]:
# Cellule 7 — sauvegarde du dataset enrichi
dataset_final.to_csv("../data/processed/dataset_enrichi.csv", index=False)

print(f"Shape final enrichi : {dataset_final.shape}")
print(f"Nouvelles colonnes ajoutées : {dataset_final.shape[1] - 83}")
print("\nDataset enrichi sauvegardé dans data/processed/dataset_enrichi.csv")

Shape final enrichi : (220, 114)
Nouvelles colonnes ajoutées : 31

Dataset enrichi sauvegardé dans data/processed/dataset_enrichi.csv


# Notebook 03 — Feature Engineering

## Objectif
Ce notebook enrichit le dataset climatique et de rendement avec de nouvelles variables construites à partir des données brutes. L'objectif est de donner au modèle des informations plus pertinentes sur les conditions agronomiques plutôt que de lui fournir uniquement les mesures climatiques brutes mois par mois.

## Ce qu'on a fait

### 1. Chargement
On part du fichier `dataset_final.csv` (220 lignes, 83 colonnes) produit dans le notebook de preprocessing.

### 2. Précipitations saisonnières
On a agrégé les précipitations mensuelles en trois périodes clés correspondant au calendrier agricole du Bénin :
- **Saison 1** (mars à juillet) : première saison des pluies, principale au Sud
- **Saison 2** (septembre à novembre) : deuxième saison des pluies au Sud
- **Saison de croissance** (mai à octobre) : période de développement du maïs commune à tout le territoire

### 3. Amplitude thermique mensuelle
Pour chaque mois, on a calculé la différence entre la température maximale et la température minimale. Cette variable capture le contraste thermique journalier, plus marqué au Nord (20-25°C d'écart) qu'au Sud (7-12°C), et qui influence la photosynthèse et le développement du grain.

### 4. Stress thermique
Le maïs souffre lorsque la température maximale dépasse 35°C, en particulier durant la floraison. On a créé une variable binaire par mois (1 si temp_max > 35°C, 0 sinon) et un compteur du nombre de mois sous stress thermique dans l'année.

### 5. Variable temporelle normalisée
On a normalisé l'année entre 0 (2003) et 1 (2022) pour permettre au modèle de capturer une éventuelle tendance temporelle dans l'évolution des rendements.

### 6. Encodage du département
Le nom du département a été converti en valeur numérique via un LabelEncoder pour être utilisable par les algorithmes de machine learning.

## Résultat final

| Indicateur | Valeur |
|---|---|
| Shape initial | 220 lignes × 83 colonnes |
| Shape final | 220 lignes × 114 colonnes |
| Nouvelles variables ajoutées | 31 |

### Détail des nouvelles variables
- 3 variables de précipitations saisonnières
- 12 amplitudes thermiques mensuelles + 1 annuelle
- 12 variables de stress thermique mensuel + 1 compteur annuel
- 1 variable temporelle normalisée
- 1 encodage département

Le dataset enrichi est sauvegardé dans `data/processed/dataset_enrichi.csv` et servira de base pour l'entraînement des modèles dans le notebook `04_model_training.ipynb`.